In [12]:
import pandas as pd
import numpy as np

crdc = pd.read_csv('../data/crdc_ap_2021_raw.csv',
                   dtype={'ncessch': str, 'crdc_id': str, 'leaid': str})
ccd = pd.read_csv('../data/ccd_directory_2021_raw.csv',
                  dtype={'ncessch': str, 'leaid': str,
                         'state_leg_district_lower': str, 'state_leg_district_upper': str})
grades = pd.read_csv('../data/ccd_enrollment_grade9to12_2021_raw.csv',
                     dtype={'ncessch': str, 'leaid': str})

print('crdc:', crdc.shape)
print('ccd:', ccd.shape)
print('grades:', grades.shape)

crdc: (98010, 17)
ccd: (102130, 52)
grades: (111511, 9)


In [13]:
# 1: in CCD, drop rows with missing enrollment, then add up the four grades for each school
grades = grades[grades['enrollment'].notna()]
enrl_9to12 = grades.groupby('ncessch')['enrollment'].sum().reset_index()
enrl_9to12.columns = ['ncessch', 'enrl_9to12']

print(enrl_9to12.shape)
print(enrl_9to12.head())

(28843, 2)
        ncessch  enrl_9to12
0  010000500871      1665.0
1  010000600872       365.0
2  010000600878       659.0
3  010000600883       413.0
4  010000600986         0.0


In [22]:
# 2: CRDC: dropped schools without a school ID (831). For repeated school IDs, kept the first record.
print('rows without a school ID:', crdc['ncessch'].isna().sum())
crdc = crdc[crdc['ncessch'].notna()]

print('duplicated school IDs:', crdc['ncessch'].duplicated().sum())
crdc = crdc.drop_duplicates(subset='ncessch')

crdc = crdc[['ncessch', 'enrl_AP']]
print(crdc.shape)

rows without a school ID: 0
duplicated school IDs: 0
(97010, 2)


In [15]:
# 3: CCD: kept schools in the 50 states, DC and Puerto Rico, offering grade 12 (including K-12 schools)
# regular schools only, and not full virtual (19,850 schools).
print('all schools:', ccd.shape)

ccd = ccd[(ccd['fips'] <= 56) | (ccd['fips'] == 72)]
print('50 states + DC + Puerto Rico:', ccd.shape)

ccd = ccd[ccd['highest_grade_offered'] == 12]
print('offers grade 12:', ccd.shape)

ccd = ccd[ccd['school_type'] == 1]
print('regular schools:', ccd.shape)

ccd = ccd[ccd['virtual'] != 1]
print('not full virtual:', ccd.shape)

all schools: (102130, 52)
50 states + DC + Puerto Rico: (101825, 52)
offers grade 12: (27560, 52)
regular schools: (20554, 52)
not full virtual: (19850, 52)


In [16]:
# 4: Merged the three files on school ID (19,561 schools)
df = ccd.merge(crdc, how='inner', on='ncessch')
print('after merging CRDC:', df.shape)

df = df.merge(enrl_9to12, how='inner', on='ncessch')
print('after merging grade 9-12 enrollment:', df.shape)

after merging CRDC: (19591, 53)
after merging grade 9-12 enrollment: (19561, 54)


In [17]:
# 5: AP participation rate = AP enrollment / grade 9–12 enrollment. 
# Schools without AP were set to 0. Dropped schools with missing AP data, zero grade 9–12 enrollment, or a rate above 1.

print(df['enrl_AP'][df['enrl_AP'] < 0].value_counts())

df['enrl_AP'] = df['enrl_AP'].replace(-2, 0)   # no AP schools
df = df[df['enrl_AP'] >= 0]                    # drop missing (-1)
print('after dropping missing AP:', df.shape)

df = df[df['enrl_9to12'] > 0]
print('after dropping zero grade 9-12 enrollment:', df.shape)

df['ap_rate'] = df['enrl_AP'] / df['enrl_9to12']
print('schools with a rate above 1:', (df['ap_rate'] > 1).sum())
df = df[df['ap_rate'] <= 1]
print('final:', df.shape)

print('fraction of schools with no AP:', (df['ap_rate'] == 0).mean())

enrl_AP
-2    6530
-1       1
Name: count, dtype: int64
after dropping missing AP: (19560, 54)
after dropping zero grade 9-12 enrollment: (19425, 54)
schools with a rate above 1: 9
final: (19416, 55)
fraction of schools with no AP: 0.32972805933250926


In [18]:
# 6: Replaced negative codes with NaN. Added two features: free/reduced-price lunch fraction and student-teacher ratio.

df['charter'] = df['charter'].replace(-2, np.nan)
df['magnet'] = df['magnet'].replace([-1, -2], np.nan)
df['title_i_status'] = df['title_i_status'].replace(-1, np.nan)

df['frl_pct'] = df['free_or_reduced_price_lunch'] / df['enrollment']

# a fraction above 1 is impossible, treat it as missing
df.loc[df['frl_pct'] > 1, 'frl_pct'] = np.nan

# 0 teachers is not realistic, treat it as missing
df['teachers_fte'] = df['teachers_fte'].replace(0, np.nan)
df['student_teacher_ratio'] = df['enrollment'] / df['teachers_fte']

print(df[['frl_pct', 'student_teacher_ratio']].describe())

            frl_pct  student_teacher_ratio
count  16071.000000           18905.000000
mean       0.499285              15.953046
std        0.268608              11.312519
min        0.001702               0.150000
25%        0.287586              11.936000
50%        0.459854              15.052632
75%        0.703011              18.571429
max        1.000000             785.000000


In [21]:
# 7: Final data: 19,416 schools, 20 columns, saved as data/ap_clean_2021.csv
columns = ['ncessch', 'school_name', 'leaid', 'state_location', 'fips',
           'latitude', 'longitude', 'urban_centric_locale',
           'lowest_grade_offered', 'charter', 'magnet', 'title_i_status',
           'enrollment', 'enrl_9to12', 'teachers_fte', 'student_teacher_ratio',
           'free_or_reduced_price_lunch', 'frl_pct',
           'enrl_AP', 'ap_rate']
df_clean = df[columns]

print(df_clean.shape)
print(df_clean.isna().mean().round(3))

df_clean.to_csv('../data/ap_clean_2021.csv', index=False)

(19416, 20)
ncessch                        0.000
school_name                    0.000
leaid                          0.000
state_location                 0.000
fips                           0.000
latitude                       0.000
longitude                      0.000
urban_centric_locale           0.000
lowest_grade_offered           0.000
charter                        0.042
magnet                         0.320
title_i_status                 0.089
enrollment                     0.000
enrl_9to12                     0.000
teachers_fte                   0.026
student_teacher_ratio          0.026
free_or_reduced_price_lunch    0.172
frl_pct                        0.172
enrl_AP                        0.000
ap_rate                        0.000
dtype: float64
